# Pesquisa bibliográfica — técnicas de normalização no treinamento de redes neurais

**Universidade Federal Rural de Pernambuco — Departamento de Estatística e Informática**
**Bacharelado em Sistemas de Informação — Projetos para Sociedade II**
**Discente:** Alexandre de Siqueira Melo Junior

Este notebook registra, de forma reproduzível, a pesquisa bibliográfica do TCC: o protocolo (questão, bases, string e critérios), a execução da busca, a validação com o *seed set*, a triagem por metadados e o resumo dos resultados.

> **Como executar:** abra no Google Colab (*Arquivo → Fazer upload de notebook*) e use *Ambiente de execução → Executar tudo*. A busca piloto usa a API aberta do OpenAlex. Sem chave, ela usa uma cota diária gratuita compartilhada pelo IP — que, no Colab, pode já estar esgotada por outros usuários. Para evitar isso, crie uma chave gratuita em [openalex.org/settings/api](https://openalex.org/settings/api) e salve-a nos *Secrets* do Colab (ícone de chave na barra lateral) com o nome `OPENALEX_API_KEY`, liberando o acesso do notebook. As exportações das bases restritas (Scopus, IEEE Xplore, Web of Science), obtidas pelo Periódicos CAPES, são opcionais e entram na seção 7.

## 1. Tema e questão de pesquisa

**Tema:** impacto das técnicas de normalização sobre o treinamento de modelos de Inteligência Artificial, com foco em redes neurais.

**Problema:** a técnica de normalização escolhida pode alterar a velocidade de convergência, a estabilidade do treinamento, o desempenho preditivo e o custo computacional; esse efeito depende da arquitetura, do tamanho do lote e do conjunto de dados.

**Questão principal (QP2):** como Batch Normalization, Layer Normalization e Group Normalization, comparadas à ausência de normalização, afetam a acurácia, a convergência, a estabilidade e o custo do treinamento de uma CNN de classificação de imagens, sob diferentes tamanhos de lote?

**Objetivo desta pesquisa bibliográfica:** identificar estudos revisados por pares, publicados em periódicos, que avaliem empiricamente esse efeito.

## 2. Protocolo

### 2.1 Bases

| Base | Tipo | Acesso | Papel |
| --- | --- | --- | --- |
| IEEE Xplore | Específica (Computação) | Periódicos CAPES | Principal |
| ACM Digital Library | Específica (Computação) | Periódicos CAPES | Principal |
| Scopus | Genérica, indexadora | Periódicos CAPES | Principal |
| Web of Science | Genérica, indexadora | Periódicos CAPES | Complementar |
| SciELO | Genérica, literatura em português | Aberto | Complementar |
| OpenAlex | Genérica, indexadora aberta | Aberto (API) | Busca piloto reproduzível neste notebook |

As bases específicas da Computação reduzem falsos positivos (o termo *normalization* tem outros sentidos em Química e Medicina). A Scopus amplia a cobertura para periódicos de outras editoras, como *Neural Networks* e *Neurocomputing*. A SciELO cobre a produção em português. O OpenAlex é aberto e permite que qualquer pessoa reexecute a busca piloto sem credenciais.

### 2.2 String de busca

A string combina dois blocos obrigatórios — **intervenção** (técnica de normalização) e **modelo** — no título, resumo e palavras-chave. Termos de desfecho e de desenho experimental não entram na string, para não excluir estudos relevantes; eles são verificados na triagem (critério I3).

São testadas duas versões:

- **String ampla** (Atividade 3): inclui também a normalização dos dados de entrada (Min-Max, Z-Score, *standardization*) e termos genéricos como *machine learning*.
- **String do recorte** (QP2): restrita à normalização interna das redes e a termos de modelo ligados a CNN e classificação de imagens. É a string usada na triagem, porque a QP2 é a questão principal do TCC.

### 2.3 Critérios (somente metadados)

| Código | Critério | Decisão |
| --- | --- | --- |
| I1 | Publicação revisada por pares em periódico. | Incluir somente se sim. |
| I2 | Periódico com Qualis A1, A2, A3, A4 ou B1 (consulta pelo ISSN). | Incluir somente se sim. |
| I3 | Título, resumo ou palavras-chave indicam experimento, comparação, *benchmark* ou avaliação. | Incluir somente se sim. |
| E1 | *Preprint*, editorial, capítulo, tese, revisão ou tutorial. | Excluir. |
| E2 | Sem ISSN ou sem Qualis elegível. | Excluir na triagem estrita. |
| E3 | Sem indício explícito de avaliação empírica. | Excluir. |

**Regra de decisão:** incluir quando `I1 AND I2 AND I3`. Se o Qualis não puder ser verificado, o registro fica **pendente**, nunca aprovado por suposição.

## 3. Configuração

In [1]:
import os
import re
import time
from pathlib import Path

import pandas as pd
import requests

pd.set_option("display.max_colwidth", 90)

ANO_INICIAL = 2015          # ano do artigo de Batch Normalization
ANO_FINAL = 2026
MAX_REGISTROS = 300         # quantos registros da busca piloto baixar para triagem
PASTA_EXPORTACOES = Path("exportacoes")   # CSVs exportados das bases CAPES (opcional)
ARQUIVO_QUALIS = Path("qualis.csv")       # lista Qualis exportada da Plataforma Sucupira (opcional)
ESTRATOS_ELEGIVEIS = {"A1", "A2", "A3", "A4", "B1"}
def ler_chave_openalex():
    # Chave gratuita opcional: Secrets do Colab ou variável de ambiente. Sem chave, usa a cota diária do IP.
    try:
        from google.colab import userdata
        return userdata.get("OPENALEX_API_KEY") or ""
    except Exception:
        return os.environ.get("OPENALEX_API_KEY", "")


OPENALEX_API_KEY = ler_chave_openalex()
print("Chave do OpenAlex:", "configurada" if OPENALEX_API_KEY else "não configurada (usa a cota gratuita do IP)")

BLOCO_INTERVENCAO = [
    "data normalization", "feature scaling", "min-max scaling", "standardization", "z-score",
    "batch normalization", "layer normalization", "group normalization",
    "filter response normalization", "normalizer-free", "normalization-free", "without normalization",
]
BLOCO_MODELO = ["neural network*", "deep learning", "machine learning", "CNN", "MLP"]

# Recorte da QP2: normalização interna em CNN de classificação de imagens
BLOCO_INTERVENCAO_RECORTE = [
    "batch normalization", "layer normalization", "group normalization", "instance normalization",
    "weight normalization", "filter response normalization", "normalizer-free", "normalization-free",
    "without normalization",
]
BLOCO_MODELO_RECORTE = ["neural network*", "deep learning", "CNN", "convolutional", "residual network*",
                        "image classification", "image recognition"]
TERMOS_EMPIRICOS = r"\b(empiric\w*|experiment\w*|benchmark\w*|compar\w*|evaluat\w*)"

SEED_SET = {
    "S1": "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift",
    "S2": "Layer Normalization",
    "S3": "Group Normalization",
    "S4": "Filter Response Normalization Layer: Eliminating Batch Dependence in the Training of Deep Neural Networks",
    "S5": "High-Performance Large-Scale Image Recognition Without Normalization",
}

Chave do OpenAlex: configurada


## 4. Funções

Cada função faz uma etapa do protocolo. A string é montada a partir dos blocos, para que a mesma definição sirva às bases CAPES (com curinga `*`) e ao OpenAlex (que não aceita curinga, mas aplica radicalização automaticamente).

In [2]:
def montar_string(blocos, curinga=True):
    # Une termos com OR dentro do bloco e blocos com AND.
    partes = []
    for bloco in blocos:
        termos = []
        for termo in bloco:
            if not curinga:
                termo = termo.replace("*", "")
            termos.append(f'"{termo}"' if (" " in termo or "-" in termo) else termo)
        partes.append("(" + " OR ".join(termos) + ")")
    return " AND ".join(partes)


def reconstruir_resumo(indice_invertido):
    # O OpenAlex entrega o resumo como índice invertido {palavra: [posições]}.
    if not indice_invertido:
        return ""
    posicoes = [(p, palavra) for palavra, lista in indice_invertido.items() for p in lista]
    return " ".join(palavra for _, palavra in sorted(posicoes))


def normalizar_doi(doi):
    if not isinstance(doi, str) or not doi.strip():
        return None
    return re.sub(r"^https?://(dx\.)?doi\.org/", "", doi.strip(), flags=re.I).lower()


def normalizar_issn(issn):
    if not isinstance(issn, str):
        return None
    digitos = re.sub(r"[^0-9Xx]", "", issn).upper()
    return f"{digitos[:4]}-{digitos[4:]}" if len(digitos) == 8 else None


def lista_issn(valor):
    # Aceita lista, string com separadores ou vazio.
    if isinstance(valor, list):
        itens = valor
    elif isinstance(valor, str):
        itens = re.split(r"[;,\s]+", valor)
    else:
        itens = []
    return sorted({i for i in (normalizar_issn(x) for x in itens) if i})


def chave_titulo(titulo):
    return re.sub(r"[^a-z0-9]", "", str(titulo).lower())


def deduplicar(df):
    # Remove duplicatas por DOI e, quando não há DOI, pelo título normalizado.
    df = df.copy()
    df["_chave"] = df["doi"].fillna("titulo:" + df["titulo"].map(chave_titulo))
    df = df.drop_duplicates("_chave").drop(columns="_chave")
    return df.reset_index(drop=True)


TIPOS_EXCLUIDOS = {"preprint", "posted-content", "editorial", "book-chapter", "dissertation",
                   "review", "letter", "erratum", "paratext", "book"}


def criterio_i1(tipo, tipo_fonte):
    # I1/E1: artigo publicado em periódico (proxy de revisão por pares).
    tipo = str(tipo).lower()
    if tipo in TIPOS_EXCLUIDOS:
        return False
    return tipo in {"article", "journal-article"} and str(tipo_fonte).lower() == "journal"


def criterio_i3(*textos):
    # I3/E3: indício explícito de avaliação empírica em título, resumo ou palavras-chave.
    return bool(re.search(TERMOS_EMPIRICOS, " ".join(str(t) for t in textos if t), flags=re.I))


def criterio_i2(issns, qualis):
    # I2/E2: True/False quando há lista Qualis; None (pendente) quando não há como verificar.
    if qualis is None:
        return None
    if not issns:
        return False
    return any(qualis.get(i) in ESTRATOS_ELEGIVEIS for i in issns)


def decidir(i1, i2, i3):
    if i1 is False or i3 is False or i2 is False:
        return "excluir"
    if i2 is None:
        return "pendente (Qualis)"
    return "incluir"


def buscar_openalex(string, filtros="", maximo=200, por_pagina=100, obter=requests.get, pausa=0.5):
    # Busca paginada por page= (o cursor= exige chave de API); devolve (total encontrado, registros brutos).
    url = "https://api.openalex.org/works"
    filtro = f"title_and_abstract.search:{string}" + (f",{filtros}" if filtros else "")
    registros, total, pagina = [], 0, 1
    while len(registros) < maximo:
        params = {"filter": filtro, "per-page": por_pagina, "page": pagina}
        if OPENALEX_API_KEY:
            params["api_key"] = OPENALEX_API_KEY
        for tentativa in range(6):
            resposta = obter(url, params=params, timeout=60)
            if resposta.status_code != 429:
                break
            espera = float(resposta.headers.get("Retry-After", 2 ** tentativa))
            if espera > 60:
                raise RuntimeError(
                    f"Cota diária gratuita do OpenAlex esgotada para este IP (libera em {espera / 3600:.1f} h). "
                    "Crie uma chave gratuita em https://openalex.org/settings/api e salve-a como OPENALEX_API_KEY "
                    "(no Colab: ícone de chave na barra lateral → Secrets), depois reexecute a partir da seção 3."
                )
            time.sleep(espera)   # limite de requisições por segundo da API
        resposta.raise_for_status()
        dados = resposta.json()
        total = dados["meta"]["count"]
        registros += dados["results"]
        if not dados["results"] or len(registros) >= total:
            break
        pagina += 1
        time.sleep(pausa)
    return total, registros[:maximo]


def registro_openalex(w):
    fonte = (w.get("primary_location") or {}).get("source") or {}
    return {
        "base": "OpenAlex",
        "titulo": w.get("title") or w.get("display_name") or "",
        "ano": w.get("publication_year"),
        "veiculo": fonte.get("display_name"),
        "tipo": w.get("type"),
        "tipo_fonte": fonte.get("type"),
        "doi": normalizar_doi(w.get("doi")),
        "issn": lista_issn(fonte.get("issn") or []),
        "resumo": reconstruir_resumo(w.get("abstract_inverted_index")),
        "palavras_chave": "; ".join(k.get("display_name", "") for k in w.get("keywords") or []),
        "citacoes": w.get("cited_by_count"),
    }

## 5. Testes das funções

Antes de usar as funções na busca, cada uma é verificada com um caso conhecido. Se algum `assert` falhar, a execução para aqui.

In [3]:
def teste(nome, condicao):
    assert condicao, nome
    print("ok —", nome)

teste("given blocos com curinga, when montar para base CAPES, then termos compostos ficam entre aspas e blocos unidos por AND",
      montar_string([["batch normalization", "CNN"], ["neural network*"]]) == '("batch normalization" OR CNN) AND ("neural network*")')
teste("given termo com curinga, when montar para OpenAlex, then o asterisco é removido",
      montar_string([["neural network*"]], curinga=False) == '("neural network")')
teste("given índice invertido, when reconstruir resumo, then palavras voltam à ordem original",
      reconstruir_resumo({"normalization": [1], "batch": [0], "helps": [2]}) == "batch normalization helps")
teste("given DOI em URL, when normalizar, then sobra só o identificador em minúsculas",
      normalizar_doi("https://doi.org/10.1109/CVPR.2020.01125") == "10.1109/cvpr.2020.01125")
teste("given ISSN sem hífen e lista mista, when normalizar, then formato NNNN-NNNN sem repetição",
      lista_issn("09252312; 0925-2312, 1872-8286") == ["0925-2312", "1872-8286"])
teste("given mesmo DOI em duas bases, when deduplicar, then sobra um registro",
      len(deduplicar(pd.DataFrame({"doi": ["10.1/a", "10.1/a", None], "titulo": ["A", "A", "B"]}))) == 2)
teste("given preprint, when avaliar I1, then é excluído",
      criterio_i1("preprint", "repository") is False)
teste("given artigo em periódico, when avaliar I1, then é aceito",
      criterio_i1("article", "journal") is True)
teste("given resumo com 'we compare', when avaliar I3, then há indício empírico",
      criterio_i3("Title", "We compare three methods"))
teste("given resumo sem termos empíricos, when avaliar I3, then não há indício",
      not criterio_i3("A survey of normalization", "This tutorial explains"))
teste("given lista Qualis com estrato A1, when avaliar I2, then é elegível",
      criterio_i2(["0925-2312"], {"0925-2312": "A1"}) is True)
teste("given lista Qualis com estrato B3, when avaliar I2, then não é elegível",
      criterio_i2(["0000-0000"], {"0000-0000": "B3"}) is False)
teste("given sem lista Qualis, when avaliar I2, then fica pendente",
      criterio_i2(["0925-2312"], None) is None)
teste("given I1 e I3 verdadeiros e I2 pendente, when decidir, then registro fica pendente",
      decidir(True, None, True) == "pendente (Qualis)")
teste("given I3 falso, when decidir, then registro é excluído",
      decidir(True, True, False) == "excluir")


class RespostaFalsa:
    def __init__(self, status, dados=None):
        self.status_code, self._dados, self.headers = status, dados, {}
    def json(self):
        return self._dados
    def raise_for_status(self):
        if self.status_code >= 400:
            raise requests.HTTPError(f"{self.status_code}")


def api_sem_cursor(url, params, timeout):
    # Simula o OpenAlex sem chave: recusa cursor, aceita page (100 registros no total).
    if "cursor" in params:
        return RespostaFalsa(401)
    inicio = (params["page"] - 1) * params["per-page"]
    lote = [{"id": i} for i in range(inicio, min(inicio + params["per-page"], 100))]
    return RespostaFalsa(200, {"meta": {"count": 100}, "results": lote})

total, achados = buscar_openalex("x", maximo=60, por_pagina=25, obter=api_sem_cursor, pausa=0)
teste("given API que recusa cursor sem chave, when buscar 60 registros, then pagina por page e devolve 60",
      total == 100 and len(achados) == 60)
total, achados = buscar_openalex("x", maximo=500, por_pagina=25, obter=api_sem_cursor, pausa=0)
teste("given menos resultados que o máximo, when buscar, then para no fim sem erro",
      total == 100 and len(achados) == 100)


def api_sem_cota(url, params, timeout):
    # Simula o OpenAlex com a cota diária esgotada: pede para esperar horas.
    resposta = RespostaFalsa(429)
    resposta.headers = {"Retry-After": "32477"}
    return resposta

inicio = time.time()
try:
    buscar_openalex("x", obter=api_sem_cota, pausa=0)
    mensagem = ""
except RuntimeError as erro:
    mensagem = str(erro)
teste("given cota diária esgotada, when buscar, then falha na hora pedindo chave de API",
      "OPENALEX_API_KEY" in mensagem and time.time() - inicio < 5)

ok — given blocos com curinga, when montar para base CAPES, then termos compostos ficam entre aspas e blocos unidos por AND
ok — given termo com curinga, when montar para OpenAlex, then o asterisco é removido
ok — given índice invertido, when reconstruir resumo, then palavras voltam à ordem original
ok — given DOI em URL, when normalizar, then sobra só o identificador em minúsculas
ok — given ISSN sem hífen e lista mista, when normalizar, then formato NNNN-NNNN sem repetição
ok — given mesmo DOI em duas bases, when deduplicar, then sobra um registro
ok — given preprint, when avaliar I1, then é excluído
ok — given artigo em periódico, when avaliar I1, then é aceito
ok — given resumo com 'we compare', when avaliar I3, then há indício empírico
ok — given resumo sem termos empíricos, when avaliar I3, then não há indício
ok — given lista Qualis com estrato A1, when avaliar I2, then é elegível
ok — given lista Qualis com estrato B3, when avaliar I2, then não é elegível
ok — given sem lista Q

## 6. Busca piloto no OpenAlex

As duas strings são aplicadas ao título e ao resumo, com os mesmos filtros adotados nas bases CAPES: artigos de periódico, de 2015 a 2026. Os registros baixados para triagem vêm da string do recorte, ordenados por relevância.

In [4]:
STRINGS = {
    "ampla": ([BLOCO_INTERVENCAO, BLOCO_MODELO]),
    "recorte": ([BLOCO_INTERVENCAO_RECORTE, BLOCO_MODELO_RECORTE]),
}
FILTROS = f"type:article,primary_location.source.type:journal,from_publication_date:{ANO_INICIAL}-01-01,to_publication_date:{ANO_FINAL}-12-31"

for nome, blocos in STRINGS.items():
    print(f"String {nome}, para as bases CAPES:\n", montar_string(blocos, curinga=True), "\n")

data_busca = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")
contagens = []
for nome, blocos in STRINGS.items():
    string = montar_string(blocos, curinga=False)
    sem_filtro, _ = buscar_openalex(string, maximo=1, por_pagina=1)
    filtrado, _ = buscar_openalex(string, FILTROS, maximo=1, por_pagina=1)
    contagens.append({"string": nome, "sem filtros": sem_filtro, f"periódicos {ANO_INICIAL}–{ANO_FINAL}": filtrado})

STRING_OPENALEX = montar_string(STRINGS["recorte"], curinga=False)
total_filtrado, brutos = buscar_openalex(STRING_OPENALEX, FILTROS, maximo=MAX_REGISTROS)
total_sem_filtro = contagens[1]["sem filtros"]

print(f"Data da busca: {data_busca}")
print(f"Registros baixados para triagem (string do recorte, mais relevantes): {len(brutos)}")
pd.DataFrame(contagens)

String ampla, para as bases CAPES:
 ("data normalization" OR "feature scaling" OR "min-max scaling" OR standardization OR "z-score" OR "batch normalization" OR "layer normalization" OR "group normalization" OR "filter response normalization" OR "normalizer-free" OR "normalization-free" OR "without normalization") AND ("neural network*" OR "deep learning" OR "machine learning" OR CNN OR MLP) 

String recorte, para as bases CAPES:
 ("batch normalization" OR "layer normalization" OR "group normalization" OR "instance normalization" OR "weight normalization" OR "filter response normalization" OR "normalizer-free" OR "normalization-free" OR "without normalization") AND ("neural network*" OR "deep learning" OR CNN OR convolutional OR "residual network*" OR "image classification" OR "image recognition") 

Data da busca: 2026-10-15 01:29
Registros baixados para triagem (string do recorte, mais relevantes): 300


,string,sem filtros,periódicos 2015–2026
0,ampla,60752,30614
1,recorte,7380,3400


## 7. Validação com o *seed set*

Uma string adequada deve recuperar os trabalhos já conhecidos. Para cada *seed*, a busca combina cada string com o título do trabalho, **sem** o filtro de periódico (os *seeds* são, em sua maioria, anais de conferência ou *preprints*).

In [5]:
def recupera_seed(string, titulo):
    filtro_titulo = f"title.search:{titulo.replace(':', '').replace(',', '')}"
    _, achados = buscar_openalex(string, filtro_titulo, maximo=5, por_pagina=5)
    alvo = chave_titulo(titulo.replace("-", " "))
    return any(alvo == chave_titulo(str(w.get("title")).replace("-", " ")) for w in achados)


linhas = []
for codigo, titulo in SEED_SET.items():
    linha = {"seed": codigo, "trabalho": titulo}
    for nome, blocos in STRINGS.items():
        linha[f"string {nome}"] = "sim" if recupera_seed(montar_string(blocos, curinga=False), titulo) else "não"
    linhas.append(linha)

seeds = pd.DataFrame(linhas)
seeds

,seed,trabalho,string ampla,string recorte
0,S1,Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate...,sim,sim
1,S2,Layer Normalization,sim,sim
2,S3,Group Normalization,sim,sim
3,S4,Filter Response Normalization Layer: Eliminating Batch Dependence in the Training of D...,sim,sim
4,S5,High-Performance Large-Scale Image Recognition Without Normalization,não,sim


## 8. Exportações das bases CAPES (opcional)

Para incluir Scopus, IEEE Xplore e Web of Science, exporte os resultados de cada base em CSV (ou texto delimitado por tabulação, no caso da Web of Science) e coloque os arquivos na pasta `exportacoes/`, com o nome da base no nome do arquivo — por exemplo, `scopus.csv`, `ieee.csv`, `wos.txt`. No Colab, use o painel *Arquivos* para enviar. Sem arquivos, esta etapa é ignorada.

In [7]:
MAPEAMENTOS = {
    "scopus": {"Title": "titulo", "Year": "ano", "Source title": "veiculo", "Document Type": "tipo",
               "DOI": "doi", "ISSN": "issn", "Abstract": "resumo", "Author Keywords": "palavras_chave"},
    "ieee": {"Document Title": "titulo", "Publication Year": "ano", "Publication Title": "veiculo",
             "DOI": "doi", "ISSN": "issn", "ISSNs": "issn", "Abstract": "resumo", "Author Keywords": "palavras_chave"},
    "wos": {"TI": "titulo", "PY": "ano", "SO": "veiculo", "DT": "tipo", "DI": "doi", "SN": "issn",
            "AB": "resumo", "DE": "palavras_chave"},
}
TIPOS_PERIODICO = {"article", "journal article", "journals"}


def carregar_exportacao(caminho):
    nome = caminho.name.lower()
    base = next((b for b in MAPEAMENTOS if b in nome), None)
    if base is None:
        print(f"Ignorado (base não reconhecida pelo nome): {caminho.name}")
        return pd.DataFrame()
    separador = "\t" if base == "wos" else ","
    bruto = pd.read_csv(caminho, sep=separador, dtype=str, encoding="utf-8-sig", on_bad_lines="skip")
    df = bruto.rename(columns=MAPEAMENTOS[base])
    df = df.loc[:, ~df.columns.duplicated()]
    for coluna in ["titulo", "ano", "veiculo", "tipo", "doi", "issn", "resumo", "palavras_chave"]:
        if coluna not in df:
            df[coluna] = None
    df["base"] = base
    df["doi"] = df["doi"].map(normalizar_doi)
    df["issn"] = df["issn"].map(lista_issn)
    df["tipo"] = df["tipo"].fillna("article").str.lower()   # IEEE só exporta o que foi filtrado como periódico
    df["tipo_fonte"] = df["tipo"].map(lambda t: "journal" if t in TIPOS_PERIODICO else t)
    df["tipo"] = df["tipo"].map(lambda t: "article" if t in TIPOS_PERIODICO else t)
    return df[["base", "titulo", "ano", "veiculo", "tipo", "tipo_fonte", "doi", "issn", "resumo", "palavras_chave"]]


exportacoes = [carregar_exportacao(p) for p in sorted(PASTA_EXPORTACOES.glob("*")) if p.is_file()] if PASTA_EXPORTACOES.exists() else []
exportacoes = [e for e in exportacoes if not e.empty]
for e in exportacoes:
    print(f"{e['base'].iloc[0]}: {len(e)} registros")
if not exportacoes:
    print("Nenhuma exportação encontrada; a triagem usará apenas a busca piloto do OpenAlex.")

Nenhuma exportação encontrada; a triagem usará apenas a busca piloto do OpenAlex.


## 9. Lista Qualis (opcional)

O critério I2 exige a classificação Qualis do periódico. Exporte a lista da área adotada pela disciplina na [Plataforma Sucupira](https://sucupira.capes.gov.br/sucupira/public/consultas/coleta/veiculoPublicacaoQualis/listaConsultaGeralPeriodicos.jsf) e salve como `qualis.csv`, com colunas de ISSN e estrato. Sem o arquivo, I2 fica **pendente** para todos os registros, como prevê o protocolo.

In [8]:
def carregar_qualis(caminho):
    if not caminho.exists():
        return None
    for codificacao in ("utf-8-sig", "latin-1"):
        try:
            tabela = pd.read_csv(caminho, sep=None, engine="python", dtype=str, encoding=codificacao)
            break
        except UnicodeDecodeError:
            continue
    col_issn = next(c for c in tabela.columns if "issn" in c.lower())
    col_estrato = next(c for c in tabela.columns if "estrato" in c.lower() or "qualis" in c.lower())
    return {normalizar_issn(i): str(e).strip().upper() for i, e in zip(tabela[col_issn], tabela[col_estrato]) if normalizar_issn(i)}


QUALIS = carregar_qualis(ARQUIVO_QUALIS)
print("Lista Qualis carregada:", f"{len(QUALIS)} periódicos" if QUALIS else "não (I2 ficará pendente)")

Lista Qualis carregada: não (I2 ficará pendente)


## 10. Consolidação, remoção de duplicatas e triagem

In [9]:
registros = pd.DataFrame([registro_openalex(w) for w in brutos])
consolidado = pd.concat([registros, *exportacoes], ignore_index=True)
total_antes = len(consolidado)
consolidado = deduplicar(consolidado)

consolidado["I1"] = [criterio_i1(t, f) for t, f in zip(consolidado["tipo"], consolidado["tipo_fonte"])]
consolidado["I2"] = [criterio_i2(i, QUALIS) for i in consolidado["issn"]]
consolidado["I3"] = [criterio_i3(t, r, k) for t, r, k in zip(consolidado["titulo"], consolidado["resumo"], consolidado["palavras_chave"])]
consolidado["decisao"] = [decidir(a, b, c) for a, b, c in zip(consolidado["I1"], consolidado["I2"], consolidado["I3"])]

print(f"Registros antes da remoção de duplicatas: {total_antes}")
print(f"Registros após a remoção de duplicatas: {len(consolidado)}")
consolidado["decisao"].value_counts().rename("registros").to_frame()

Registros antes da remoção de duplicatas: 300
Registros após a remoção de duplicatas: 299


,registros
decisao,
pendente (Qualis),178
excluir,121


## 11. Resultados

### 11.1 Fluxo da seleção

In [10]:
fluxo = pd.DataFrame([
    ("Encontrados no OpenAlex (string do recorte), sem filtros", total_sem_filtro),
    (f"Encontrados no OpenAlex (string do recorte), periódicos {ANO_INICIAL}–{ANO_FINAL}", total_filtrado),
    ("Baixados para triagem (OpenAlex)", len(registros)),
    ("Importados das bases CAPES", sum(len(e) for e in exportacoes)),
    ("Após remoção de duplicatas", len(consolidado)),
    ("Excluídos por I1/E1 (não é artigo de periódico)", int((~consolidado["I1"]).sum())),
    ("Excluídos por I3/E3 (sem indício empírico)", int((consolidado["I1"] & ~consolidado["I3"]).sum())),
    ("Incluídos", int((consolidado["decisao"] == "incluir").sum())),
    ("Pendentes de verificação Qualis", int((consolidado["decisao"] == "pendente (Qualis)").sum())),
], columns=["etapa", "registros"])
fluxo

,etapa,registros
0,"Encontrados no OpenAlex (string do recorte), sem filtros",7380
1,"Encontrados no OpenAlex (string do recorte), periódicos 2015–2026",3400
2,Baixados para triagem (OpenAlex),300
3,Importados das bases CAPES,0
4,Após remoção de duplicatas,299
5,Excluídos por I1/E1 (não é artigo de periódico),0
6,Excluídos por I3/E3 (sem indício empírico),121
7,Incluídos,0
8,Pendentes de verificação Qualis,178


### 11.2 Técnicas de normalização mais citadas nos registros aprovados ou pendentes

In [11]:
candidatos = consolidado[consolidado["decisao"] != "excluir"]
texto = (candidatos["titulo"].fillna("") + " " + candidatos["resumo"].fillna("")).str.lower()
tecnicas = {
    "Batch Normalization": r"batch[- ]normali[sz]",
    "Layer Normalization": r"layer[- ]normali[sz]",
    "Group Normalization": r"group[- ]normali[sz]",
    "Instance Normalization": r"instance[- ]normali[sz]",
    "Weight Normalization": r"weight[- ]normali[sz]",
    "Filter Response Normalization": r"filter response normali[sz]",
    "Sem normalização / normalizer-free": r"normali[sz]er[- ]free|without normali[sz]",
    "Min-Max / Z-Score (dados de entrada)": r"min[- ]max|z[- ]score|standardi[sz]ation",
}
pd.DataFrame(
    {"registros": {nome: int(texto.str.contains(padrao).sum()) for nome, padrao in tecnicas.items()}}
).sort_values("registros", ascending=False)

,registros
Batch Normalization,167
Layer Normalization,5
Instance Normalization,4
Weight Normalization,4
Min-Max / Z-Score (dados de entrada),3
Sem normalização / normalizer-free,3
Group Normalization,1
Filter Response Normalization,0


### 11.3 Registros por ano

In [12]:
candidatos.groupby("ano").size().rename("registros").to_frame().T

ano,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
registros,1,1,6,7,22,19,17,17,27,27,17,17


### 11.4 Periódicos com mais registros

In [13]:
candidatos["veiculo"].value_counts().head(10).rename("registros").to_frame()

,registros
veiculo,
IEEE Access,13
JOIV International Journal on Informatics Visualization,6
IEEE Transactions on Geoscience and Remote Sensing,4
Remote Sensing,4
Scientific Reports,3
Electronics,3
IEEE Transactions on Pattern Analysis and Machine Intelligence,3
IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing,3
Sensors,3


### 11.5 Registros mais citados entre os candidatos

In [14]:
colunas = ["titulo", "ano", "veiculo", "doi", "citacoes", "decisao"]
candidatos.sort_values("citacoes", ascending=False, na_position="last")[colunas].head(15)

,titulo,ano,veiculo,doi,citacoes,decisao
233,MRI-based brain tumor detection using convolutional deep learning methods and chosen m...,2023,BMC Medical Informatics and Decision Making,10.1186/s12911-023-02114-6,617,pendente (Qualis)
9,Dropout vs. batch normalization: an empirical study of their impact to deep learning,2020,Multimedia Tools and Applications,10.1007/s11042-019-08453-9,571,pendente (Qualis)
104,CNN-LSTM: Hybrid Deep Neural Network for Network Intrusion Detection System,2022,IEEE Access,10.1109/access.2022.3206425,417,pendente (Qualis)
61,SAR Image Despeckling Using a Convolutional Neural Network,2017,IEEE Signal Processing Letters,10.1109/lsp.2017.2758203,407,pendente (Qualis)
113,A Fast Dense Spectral–Spatial Convolution Network Framework for Hyperspectral Images C...,2018,Remote Sensing,10.3390/rs10071068,379,pendente (Qualis)
172,Learning Sensor-Specific Spatial-Spectral Features of Hyperspectral Images via Convolu...,2017,IEEE Transactions on Geoscience and Remote Sensing,10.1109/tgrs.2017.2693346,311,pendente (Qualis)
87,Systematic evaluation of convolution neural network advances on the Imagenet,2017,Computer Vision and Image Understanding,10.1016/j.cviu.2017.05.007,308,pendente (Qualis)
95,"Deep convolutional neural networks for mammography: advances, challenges and applications",2019,BMC Bioinformatics,10.1186/s12859-019-2823-4,271,pendente (Qualis)
11,Hyperspectral Image Classification Using a Hybrid 3D-2D Convolutional Neural Networks,2021,IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing,10.1109/jstars.2021.3099118,234,pendente (Qualis)
187,Diagnosis of cystic lesions using panoramic and cone beam computed tomographic images ...,2019,Oral Diseases,10.1111/odi.13223,230,pendente (Qualis)


## 12. Exportação

In [15]:
saida = consolidado.assign(issn=consolidado["issn"].map("; ".join))
saida.to_csv("triagem_pesquisa_bibliografica.csv", index=False)
fluxo.to_csv("fluxo_selecao.csv", index=False)
print("Arquivos gerados: triagem_pesquisa_bibliografica.csv, fluxo_selecao.csv")

Arquivos gerados: triagem_pesquisa_bibliografica.csv, fluxo_selecao.csv


## 13. Conclusões e próximos passos

- A string ampla da Atividade 3 traz muitos registros de outras áreas (aplicações médicas, por exemplo), porque termos como *standardization* e *machine learning* são genéricos. Além disso, ela não recupera o S5, cujo resumo fala em *image classifiers* e *ResNets*, e não em *neural network* ou *deep learning*.
- A string do recorte, com *convolutional*, *residual network* e *image classification* no bloco de modelo, recupera todo o *seed set* e reduz o volume a registros do tema. Ela deve substituir a string ampla nas bases CAPES.
- A triagem é objetiva e baseada apenas em metadados (`I1 AND I2 AND I3`). Os registros ficam **pendentes** até a verificação Qualis pelo ISSN.
- Limitação: muitos candidatos usam Batch Normalization apenas como componente de uma aplicação (diagnóstico por imagem, sensoriamento remoto), sem comparar técnicas de normalização. O critério I3 identifica avaliação empírica, mas não se a normalização é o objeto da comparação; essa distinção será feita na leitura dos resumos dos registros aprovados.
- O conjunto de candidatos mostra quais técnicas aparecem com mais frequência na literatura empírica em periódicos, o que confirma a comparação entre Batch, Layer e Group Normalization e a condição sem normalização.

**Próximos passos:**

1. Executar a mesma string na Scopus, no IEEE Xplore e na Web of Science pelo Periódicos CAPES e enviar as exportações para a pasta `exportacoes/`.
2. Executar a versão em português na SciELO.
3. Carregar a lista Qualis da área definida com o docente (`qualis.csv`) e reexecutar o notebook.
4. Ler na íntegra os registros incluídos e extrair técnicas, arquiteturas, *datasets*, tamanhos de lote e métricas.